# Weekly Sales Forecasting & Explainability

**Pregunta:** ¿un modelo complejo mejora la predicción de ventas de la semana siguiente?

Ejecuta primero `python main.py --source real --explain` o el notebook 01 para crear el panel semanal. La evaluación final utiliza las últimas 16 semanas completas, con historial anterior observado. No es una predicción de todo el periodo desde una única fecha.

## 1. Preparación

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from a checkout of retail-analytics-pipeline.")
sys.path.insert(0, str(ROOT / "src"))
REPORTS = ROOT / "reports"


**Dónde:** Jupyter en Codespaces o una copia del repositorio. La carpeta raíz se identifica por `pyproject.toml`; los mismos módulos se usan desde `main.py`.

## 2. Leer el panel y construir features

In [2]:
from market_insights.data import real_weekly, load_transactions
from market_insights.quality import clean_transactions
from market_insights.features import build_features, temporal_split
path = ROOT/'data/processed/weekly_real.csv'
if not path.exists():
    raise FileNotFoundError('Ejecuta main.py --source real o el notebook 01 primero.')
weekly = pd.read_csv(path, parse_dates=['week_start'], dtype={'segment':'string'})
cutoff = weekly.week_start.sort_values().unique()[-16]
features, numeric = build_features(weekly, cutoff)
train, test = temporal_split(features, cutoff)
print('Train:', len(train), '| Test:', len(test), '| Corte:', pd.Timestamp(cutoff).date())
display(train[numeric].head())

Train: 1008 | Test: 192 | Corte: 2011-08-15
   lag_1  lag_2  lag_4  ...  price_lag_1  is_high_demand  high_demand_rd
4    0.0  103.0   91.0  ...     0.694175               0               0
5  192.0    0.0   63.0  ...     0.650000               0               0
6   30.0  192.0  103.0  ...     1.028667               0               0
7  118.0   30.0    0.0  ...     0.703390               0               0
8   59.0  118.0  192.0  ...     0.767458               0               0

[5 rows x 10 columns]


**Stack:** pandas y NumPy. Lags y medias móviles usan `shift(1)`: no utilizan ventas de la semana pronosticada. El precio es el último observado, no el precio medio futuro. El umbral de demanda histórica alta se aprende con train; no identifica pedidos mayoristas verificados.

## 3. Validaciones temporales

In [3]:
from market_insights.forecasting import backtest
validation = backtest(weekly, cutoff)
display(validation)
display(validation.groupby('model')[['mae','wape']].mean())

            model         mae        r2  ...  fold  validation_start validation_end
0           naive  454.604167 -0.835123  ...     1        2011-02-28     2011-04-18
1   random_forest  356.314708  0.050377  ...     1        2011-02-28     2011-04-18
2         xgboost  368.802531  0.044180  ...     1        2011-02-28     2011-04-18
3        ensemble  358.931294  0.065313  ...     1        2011-02-28     2011-04-18
4           naive  450.250000 -0.916816  ...     2        2011-04-25     2011-06-13
5   random_forest  379.114843 -0.094134  ...     2        2011-04-25     2011-06-13
6         xgboost  377.654504 -0.137661  ...     2        2011-04-25     2011-06-13
7        ensemble  374.358719 -0.088495  ...     2        2011-04-25     2011-06-13
8           naive  389.885417 -0.568092  ...     3        2011-06-20     2011-08-08
9   random_forest  382.399543 -0.103786  ...     3        2011-06-20     2011-08-08
10        xgboost  409.604027 -0.312960  ...     3        2011-06-20     201

**Stack:** scikit-learn y XGBoost. Tres validaciones de ocho semanas ocurren antes del holdout final. El preprocesamiento y los umbrales se ajustan dentro de cada fold. Los parámetros están fijados por el ejercicio; este notebook no usa el test para tuning.

## 4. Comparar baseline, RF, XGBoost y ensemble

In [4]:
from market_insights.forecasting import fit_forecasters
metrics, predictions, fitted = fit_forecasters(train, test, numeric)
display(metrics)
display(predictions.head())

           model         mae        r2      wape
0          naive  408.453125 -0.096420  0.665690
1  random_forest  454.027327 -0.019810  0.739967
2        xgboost  517.540716 -0.408883  0.843480
3       ensemble  479.397854 -0.168105  0.781315
   week_start         country segment  ...  random_forest     xgboost    ensemble
88 2011-08-15  United Kingdom   15036  ...     522.918698  477.220520  500.069609
89 2011-08-22  United Kingdom   15036  ...     514.560282  501.543640  508.051961
90 2011-08-29  United Kingdom   15036  ...     110.641093  216.860291  163.750692
91 2011-09-05  United Kingdom   15036  ...     176.226185  199.079041  187.652613
92 2011-09-12  United Kingdom   15036  ...     156.443979  101.635887  129.039933

[5 rows x 8 columns]


**Stack:** scikit-learn y XGBoost. El baseline repite las ventas de la última semana. El ensemble es un promedio 50/50 definido antes de evaluar. MAE se mide en unidades y WAPE expresa error agregado relativo. Con estos datos registrados, la última semana supera a los modelos complejos en MAE.

![Figura de la ejecución registrada](../reports/figures/model_comparison.png)

## 5. Inspeccionar predicciones y errores

In [5]:
from market_insights.visualization import plot_forecast, plot_errors
# Elegido por volumen de entrenamiento, no por la apariencia de su curva de test.
example = train.groupby('segment').units_sold.sum().idxmax()
plot_forecast(predictions, example, 'UCI | UK training-selected SKUs', REPORTS/'figures')
plot_errors(predictions, 'UCI | final temporal holdout', REPORTS/'figures')

**Stack:** Matplotlib. Mantener valores reales y predicciones en la misma escala. Revisar qué semanas y SKU concentran el error; no atribuirlo automáticamente a promociones, stockouts o compradores mayoristas si esas variables no están disponibles.

![Figura de la ejecución registrada](../reports/figures/forecast_vs_actual.png)

## 6. SHAP del promedio de modelos

In [6]:
from market_insights.explainability import explain
shap_audit = explain(fitted, 'UCI | ensemble on the final holdout', REPORTS/'figures', REPORTS/'metrics')
display(pd.DataFrame([shap_audit]))
display(pd.read_csv(REPORTS/'metrics/shap_importance.csv').head(10))

   sample_n                  reference  max_additivity_error
0        80  training tree-path counts              0.001105
                    feature  mean_abs_shap
0         numeric__week_sin     159.600929
1  numeric__rolling_avg_4wk     103.301649
2      numeric__price_lag_1      81.261021
3            numeric__lag_4      63.907483
4  numeric__rolling_avg_2wk      58.170501
5            numeric__lag_1      45.964048
6            numeric__lag_2      24.470620
7         numeric__week_cos      22.205807
8   category__segment_84077      18.873321
9   category__segment_21212      13.724003


**Stack:** SHAP. Instalar `pip install -e '.[explain]'` si falta. Se usa la referencia de recorridos de entrenamiento de cada árbol, se promedian contribuciones y valores base, y se verifica que reconstruyan el ensemble. Las contribuciones explican el modelo; no establecen causalidad.

![Figura de la ejecución registrada](../reports/figures/shap_global.png)

## 7. Decisión y limitaciones

El resultado no justifica desplegar el ensemble como mejora del baseline. Conservar el baseline y estudiar estabilidad por SKU, periodos de inventario y nuevos datos antes de recomendar un modelo operativo. Los R² negativos quedan registrados; no se sustituyen por las métricas de la simulación.